# RQ3: FinBERT baseline

Runs `yiyanghkust/finbert-tone` on the clean base sentences (`1_clean_baseline`) and on the biased sentences (`2_no_defense`).
Writes to `outputs/bias_defense/finbert-tone/` in the same format as `bias_defense.py`, so `rq3_defense_analysis.ipynb` picks it up.

In [ ]:
import os

import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report
from transformers import BertForSequenceClassification, BertTokenizer, pipeline

DATASET_PATH = "../data/triggerbank.csv"
OUTPUT_DIR = "../outputs/bias_defense/finbert-tone"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {torch.cuda.get_device_name(0) if device == 0 else 'CPU'}")

In [ ]:
# Load model
finbert = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-tone', num_labels=3)
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-tone')
nlp = pipeline("sentiment-analysis", model=finbert, tokenizer=tokenizer, device=device, batch_size=32)

# Label mapping
LABEL_MAP = {
    "Neutral": "neutral",
    "Positive": "positive",
    "Negative": "negative"
}

def predict_sentiment(sentences):
    """Predict sentiment for a list of sentences."""
    results = nlp(sentences, truncation=True, max_length=512)
    return [LABEL_MAP[r["label"]] for r in results]

In [ ]:
# Load dataset
df = pd.read_csv(DATASET_PATH)
df_clean = df[df["biases"] == "none"].copy()
df_biased = df[df["biases"] != "none"].copy()

print(f"Clean sentences: {len(df_clean)}")
print(f"Biased sentences: {len(df_biased)}")

all_results = []

# Condition 1: Clean baseline
print("\nRunning: 1_clean_baseline")
df_clean["predicted_sentiment"] = predict_sentiment(df_clean["original_sentence"].tolist())
df_clean["parsed_sentiment"] = df_clean["predicted_sentiment"]
df_clean["condition"] = "1_clean_baseline"
all_results.append(df_clean.copy())

# Condition 2: No defense (biased sentences)
print("Running: 2_no_defense")
df_biased["predicted_sentiment"] = predict_sentiment(df_biased["biased_sentence"].tolist())
df_biased["parsed_sentiment"] = df_biased["predicted_sentiment"]
df_biased["condition"] = "2_no_defense"
all_results.append(df_biased.copy())

# Combine and save predictions
combined_df = pd.concat(all_results, ignore_index=True)
predictions_path = os.path.join(OUTPUT_DIR, "predictions_finbert-tone.csv")
combined_df.to_csv(predictions_path, index=False)
print(f"\nSaved predictions to: {predictions_path}")

In [ ]:
# Generate classification reports
all_reports = []

for condition in ["1_clean_baseline", "2_no_defense"]:
    cond_df = combined_df[combined_df["condition"] == condition]
    
    report = classification_report(
        cond_df["label"],
        cond_df["parsed_sentiment"],
        labels=["positive", "neutral", "negative"],
        output_dict=True,
        zero_division=0
    )
    
    acc = accuracy_score(cond_df["label"], cond_df["parsed_sentiment"])
    
    report_df = pd.DataFrame(report).transpose()
    report_df["condition"] = condition
    all_reports.append(report_df)
    
    print(f"\n{'='*60}")
    print(f"Condition: {condition}")
    print(f"Accuracy: {acc:.4f}")
    print(f"{'='*60}")
    print(classification_report(
        cond_df["label"],
        cond_df["parsed_sentiment"],
        labels=["positive", "neutral", "negative"],
        zero_division=0
    ))

# Save classification reports
combined_reports = pd.concat(all_reports)
report_path = os.path.join(OUTPUT_DIR, "classification_report_finbert-tone.csv")
combined_reports.to_csv(report_path, index=True)
print(f"\nSaved classification report to: {report_path}")